In [ ]:
import pandas as pd
import requests
import regex as re
from urllib.parse import urlencode,urlunsplit
import numpy as np
from bs4 import BeautifulSoup

In [ ]:
def load_students(csv):
    df = pd.read_csv(csv)
    return df, df.shape

df_habitos,_ = load_students('student_habits_performance.csv')
df_habitos

In [ ]:
def clean_students(df_habitos:pd.DataFrame):
    df = df_habitos.copy()
    df = df.drop_duplicates('student_id')
    df.loc[(df['attendance_percentage']>100)|(df['attendance_percentage']<0),'attendance_percentage'] = np.nan
    df.loc[(df['sleep_hours']>24)|(df['sleep_hours']<0),'sleep_hours'] = df['sleep_hours'].median()
    return df

df_clean = clean_students(df_habitos)
df_clean

In [ ]:
url = 'https://nces.ed.gov/programs/digest/d21/tables/dt21_227.40.asp'

def get_soup(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.content)
    return soup

soup = get_soup(url)
soup

In [ ]:
def extract_expenditure(soup:BeautifulSoup):
    tabla = soup.findAll('table', {'class':"tabletop tableMain tableWidth tableBracketRow"})
    df = pd.read_html(str(tabla))
    return df

df = extract_expenditure(soup)[0]
df

In [ ]:
SCHEME = 'https'
BASE = 'educationdata.urban.org/api/v1/'
def get_url(topic='',source='',endpoint='',year='',especifiers='',query_dict=None):
    path = f'{topic}/{source}/{endpoint}/{year}/{especifiers}/'
    if query_dict is None:
        query_dict = {}
    query = urlencode(query_dict)
    return urlunsplit((SCHEME,BASE,path,query_dict,''))

url = get_url('schools','cdd','enrollment','2021','grade-3')
url

In [ ]:
response = requests.get(url)
response.content

In [ ]:
df_habitos['study_hours_per_day'].plot(kind='box')

In [ ]:
def impute_outliers(df_habitos:pd.DataFrame):
    df = df_habitos.copy()
    df['zscore'] = abs((df['study_hours_per_day']-df['study_hours_per_day'].mean())/df['study_hours_per_day'].std())
    df.loc[df['zscore']>3,'study_hours_per_day'] = df['study_hours_per_day'].median()
    return df

df = impute_outliers(df_habitos)
df

In [ ]:
df['study_hours_per_day'].plot(kind='box')

In [ ]:
df.isna().sum()

In [ ]:
def impute_missing(df_hab:pd.DataFrame):
    df = df_hab.copy()
    df['parental_education_level'] = (df.groupby('diet_quality')['parental_education_level'].transform(lambda m: m.fillna(m.mode())))
    return df 
df_final = impute_missing(df)
df_final

In [ ]:
df_final.loc[df['parental_education_level'].isna()]

In [ ]:
df_final.isna().sum()